Import Libraries

In [52]:
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm
from node2vec import Node2Vec

File Paths

In [53]:
MERGED_PATH = "../data/processed/merged_gene_mutation_features.csv"
EDGE_PATH = "../data/processed/string_graph_edges.csv"
GENE_INFO_PATH = "../data/processed/gene_info_filtered.csv"

OUTPUT_FEATURES = "../data/processed/final_gene_features.csv"
OUTPUT_EDGES = "../data/processed/final_edge_list.csv"

Load Datasets

In [54]:
df = pd.read_csv(MERGED_PATH)
edges = pd.read_csv(EDGE_PATH)
gene_info = pd.read_csv(GENE_INFO_PATH)

print("Mutation dataset:", df.shape)
print("Edges:", edges.shape)
print("Gene info:", gene_info.shape)

Mutation dataset: (23050, 7)
Edges: (472000, 2)
Gene info: (193791, 3)


Clean Column Names

In [55]:
df.columns = df.columns.str.strip()
edges.columns = edges.columns.str.strip()
gene_info.columns = gene_info.columns.str.strip()

Standardize Gene Symbols

In [56]:
df["GeneSymbol"] = df["GeneSymbol"].str.upper()

gene_info.rename(columns={"Symbol":"GeneSymbol"}, inplace=True)
gene_info["GeneSymbol"] = gene_info["GeneSymbol"].str.upper()

Merge Gene Metadata

In [57]:
df = df.merge(
    gene_info[["GeneSymbol","description"]],
    on="GeneSymbol",
    how="left"
)

Create Missing Feature

In [58]:
df["rare_variants"] = (
    df["total_variants"] - df["benign_variants"]
)

Build STRING Graph

In [59]:
print("Building STRING network graph...")

G = nx.from_pandas_edgelist(
    edges,
    source="gene1",
    target="gene2"
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Building STRING network graph...
Nodes: 16185
Edges: 236000


Basic Graph Features

In [60]:
print("Computing graph topology features...")

degree = dict(G.degree())
clustering = nx.clustering(G)
pagerank = nx.pagerank(G)

# faster approximate betweenness
betweenness = nx.betweenness_centrality(G, k=100, seed=42)

Computing graph topology features...


Map Graph Features

In [61]:
df["gene_degree"] = df["GeneSymbol"].map(degree).fillna(0)

df["clustering_coefficient"] = df["GeneSymbol"].map(clustering).fillna(0)

df["pagerank"] = df["GeneSymbol"].map(pagerank).fillna(0)

df["betweenness_centrality"] = df["GeneSymbol"].map(betweenness).fillna(0)

Node2Vec Graph Embeddings

In [63]:
from node2vec import Node2Vec

print("Training Node2Vec embeddings...")

node2vec = Node2Vec(
    G,
    dimensions=32,
    walk_length=15,   # reduced
    num_walks=30,     # reduced
    workers=1         # Windows safe
)

model = node2vec.fit(
    window=10,
    min_count=1,
    batch_words=4
)

Training Node2Vec embeddings...


Generating walks (CPU: 1): 100%|██████████| 30/30 [01:03<00:00,  2.13s/it]


Convert Embeddings to DataFrame

In [64]:
embedding_dict = {
    node: model.wv[node]
    for node in G.nodes()
}

embedding_df = pd.DataFrame.from_dict(
    embedding_dict,
    orient="index"
)

embedding_df.index.name = "GeneSymbol"
embedding_df.reset_index(inplace=True)

Rename Embedding Columns

In [65]:
embedding_df.columns = (
    ["GeneSymbol"] +
    [f"node2vec_{i}" for i in range(32)]
)

Merge Node2Vec Features

In [66]:
df = df.merge(
    embedding_df,
    on="GeneSymbol",
    how="left"
)

df.fillna(0, inplace=True)

print("Dataset shape after Node2Vec:", df.shape)

Dataset shape after Node2Vec: (23053, 45)


Neighbor Pathogenic Signal

In [67]:
print("Computing neighbor pathogenic signal...")

pathogenic_map = dict(
    zip(df["GeneSymbol"], df["pathogenic_variants"])
)

neighbor_ratio = {}

for gene in tqdm(df["GeneSymbol"]):

    if gene not in G:
        neighbor_ratio[gene] = 0
        continue

    neighbors = list(G.neighbors(gene))

    if len(neighbors) == 0:
        neighbor_ratio[gene] = 0
        continue

    pathogenic_neighbors = sum(
        pathogenic_map.get(n,0) > 0
        for n in neighbors
    )

    neighbor_ratio[gene] = pathogenic_neighbors / len(neighbors)

df["neighbor_pathogenic_ratio"] = df["GeneSymbol"].map(neighbor_ratio)

Computing neighbor pathogenic signal...


100%|██████████| 23053/23053 [00:00<00:00, 237013.44it/s]


Interaction Features

In [68]:
df["mutation_network_score"] = (
    df["total_variants"] * df["gene_degree"]
)

df["rare_network_score"] = (
    df["rare_variants"] * df["gene_degree"]
)

Create Label

In [69]:
df["label"] = (
    df["pathogenic_variants"] > 0
).astype(int)

Remove Duplicate Columns

In [70]:
df = df.loc[:, ~df.columns.duplicated()]

Remove Duplicate Columns

In [71]:
df = df.drop_duplicates(subset=["GeneSymbol"])

print("Total genes:", df.shape[0])
print("Duplicate genes:", df["GeneSymbol"].duplicated().sum())

Total genes: 23050
Duplicate genes: 0


Save Final Dataset

In [72]:
df.to_csv(OUTPUT_FEATURES, index=False)

edges.to_csv(OUTPUT_EDGES, index=False)

print("Feature construction complete.")

Feature construction complete.
